# Corporacion Favorita - New Superb Forecasting Model - 

## Split and Model Pipeline

#codi

Made by 4B Consultancy (Janne Heuvelmans, Georgi Duev, Alexander Engelage, Sebastiaan de Bruin) - 2024

In this data pipeline, 

The following steps are made within this notebook:  

>-0. Import Packages 

>-1. Load final dataset and aggregate dataset to weekly level
    -1.1 Load final dataset made in Data Preperation Pipeline Notebook
    -1.2 Aggregate dataset to weekly level

>-2. Column transformers and Train, Test, Validation Split

>-3. Models

>-4. Pick best model one and optimize with grid search

## 0. Import Packages

In [1]:
# Importing the libraries
import pandas as pd
import numpy as np
import polars as pl
import os
import sys
import matplotlib.pyplot as plt
import altair as alt
import vegafusion as vf
import sklearn
import time
from datetime import date, datetime, timedelta
from sklearn.pipeline import Pipeline, make_pipeline

In [2]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.metrics import mean_absolute_percentage_error

import statsmodels.api as sm

In [3]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose

In [4]:
from sktime.forecasting.compose import EnsembleForecaster

## 1. Load final dataset

### 1.1. Functions - Import raw data from local PATH
Create import data function and give basic information function within the importing function.

Return basic information on each dataframe:  
- a) Information on the number of observation and features.  
- b) Information on the size of the dataframe. 

TO-DO: Import via polars, and use polars dataframe?

In [5]:
def f_get_data_and_info(import_path, file_name):

    print(f"\nReading file {file_name}\n")

    # Load data.
    df = pd.read_parquet(import_path + file_name + ".parquet")

    # Getting the basic information of the dataframe (number of observations and features, and size)
    print(
        f"The '{file_name}' dataframe contains: {df.shape[0]:,}".replace(",", ".")
        + f" observations and {df.shape[1]} features."
    )
    print(
        f"Prepared and transformed dataframe has optimized size of {round(sys.getsizeof(df)/1024/1024/1024, 2)} GB."
    )

    return df

### 1.2. Importing raw data
Importing parquet files with importing function (giving basic information)

In [ ]:
import_path = "C:/Users/alexander/Documents/0. Data Science and AI for Experts/FILE/"

# Importing final df
df_final = f_get_data_and_info(import_path, file_name="Prepped_data_20241211")

# df_final = f_get_data_and_info(import_path, file_name="df_test_with_forecasts_20241120")

## 2.0 Train Test Val Split

train_test_val_split without creating X (features) and y (target)

In [7]:
def train_test_val_split(df, window_length=26):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "week_number_cum"])

    # Get the maximum week in the dataset
    max_week = df["week_number_cum"].max()

    # Calculate start and end weeks for test and validation  sets
    val_week_end = max_week - 1

    val_week_start = max_week - window_length

    test_week_start = max_week - 2 * window_length

    test_week_end = val_week_start - 1

    train_week_start = max_week - 6 * window_length

    train_week_end = test_week_start - 1

    # Train data: All data before the start of the validation period
    train = df[
        (df["week_number_cum"] >= train_week_start)
        & (df["week_number_cum"] <= train_week_end)
    ]

    # Val data: From val_week_start to val_week_end
    test = df[
        (df["week_number_cum"] >= test_week_start)
        & (df["week_number_cum"] <= test_week_end)
    ]

    # Test data: From test_week_start to max_week
    val = df[
        (df["week_number_cum"] >= val_week_start)
        & (df["week_number_cum"] <= val_week_end)
    ]

    # Function to print split information
    def print_split_info(split_name, split):
        print(f"\n{split_name} set: shape: {split.shape}")
        print(f"{split_name} Min Week: {split['week_number_cum'].min()}")
        print(f"{split_name} Min Date: {split['date'].min()}")
        print(f"{split_name} Max Week: {split['week_number_cum'].max()}")
        print(f"{split_name} Max Date: {split['date'].max()}")
        print(f"{split_name} number of weeks: {split['week_number_cum'].nunique()}")
        print(f"Number of stores: {split['store_nbr'].nunique()}")
        print(f"Number of items: {split['item_nbr'].nunique()}")
        print(f"Size of {round(sys.getsizeof(split)/1024/1024/1024, 2)} GB.")

    # Print information about the splits
    print_split_info("Train", train)
    print_split_info("Test", test)
    print_split_info("Validation", val)

    return train, test, val

## 3.0 Functions - Impute stockouts and Aggregate dataset to weekly level


#### 3.1. Impute stockouts

Stockout on store level

•      Perishable good: when there are missing values for two consecutive days for a given item per individual store 

•      Nonperishable goods: when there are missing values for 7 consecutive days for a given item and per individual store

•      Action: Impute with Rolling Mean with defeault window of 7 days 

------------------------------------

In [8]:
def impute_stockouts_polars(df_pandas, window_size=7):
    # Convert the input Pandas DataFrame to a Polars DataFrame
    df = pl.from_pandas(df_pandas)

    # Sort the DataFrame by store number, item number, and date for proper ordering
    df = df.sort(["store_nbr", "item_nbr", "date"])

    # Create a boolean column to indicate where 'unit_sales' is missing
    df = df.with_columns((pl.col("unit_sales").is_null()).alias("is_missing"))

    # Assign a group identifier to each segment of missing or non-missing values
    df = df.with_columns(
        (
            pl.col("is_missing").cast(pl.Int16)
            != pl.col("is_missing").cast(pl.Int16).shift(1)
        )
        .cast(pl.Int16)
        .cum_sum()
        .alias("missing_group")
        .cast(pl.Int32)
    )

    # Calculate cumulative count of missing values within each segment of missing data
    df = df.with_columns(
        pl.when(pl.col("is_missing"))
        .then(
            pl.col("is_missing")
            .cast(pl.Int16)
            .cum_sum()
            .over(["store_nbr", "item_nbr", "missing_group"])
        )
        .otherwise(0)
        .alias("missing_count")
    )

    # Identify groups to find maximum of the same missing_count group
    df = df.with_columns(
        pl.col("missing_count")
        .max()
        .over(["missing_group"])
        .alias("group_max_missing_count")
        .cast(pl.Int16)
    )

    # Function for rolling mean imputation
    def rolling_mean_imputation(df, window_size=7):
        # Add rolling mean column for grouped data
        df = df.with_columns(
            pl.col("unit_sales")
            .rolling_mean(window_size=window_size, min_periods=1)
            .over(["store_nbr", "item_nbr"])
            .shift(1)  # Shift to exclude current row
            .alias("unit_sales_rolling_mean")
        )

        # Replace nulls in the target column with the calculated rolling mean
        df = df.with_columns(
            pl.when(pl.col("unit_sales").is_null())
            .then(pl.col("unit_sales_rolling_mean"))
            .otherwise(pl.col("unit_sales"))
            .alias("unit_sales")
        )

        # Drop the temporary rolling mean column
        df = df.drop("unit_sales_rolling_mean")

        return df

    # Apply rolling mean imputation based on perishable status
    df = df.with_columns(
        [
            pl.when(pl.col("perishable") == 1)  # If the item is perishable
            .then(
                pl.when(pl.col("group_max_missing_count") == 1)  # 1 missing value
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") > 2
                )  # More than 2 missing values
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") == 2
                )  # Exactly 2 missing values
                .then(
                    rolling_mean_imputation(df, window_size=7)["unit_sales"]
                )  # Impute with rolling mean for 2 missing days
                .otherwise(pl.col("unit_sales"))  # Keep original value
            )
            .when(pl.col("perishable") == 0)  # If the item is not perishable
            .then(
                pl.when(
                    pl.col("group_max_missing_count") > 7
                )  # More than 7 missing values
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") <= 7
                )  # 7 or fewer missing values
                .then(
                    rolling_mean_imputation(df, window_size=7)["unit_sales"]
                )  # Impute with rolling mean for missing 7 or fewer days
                .otherwise(pl.col("unit_sales"))  # Keep original value
            )
            .otherwise(pl.col("unit_sales"))  # For other cases, keep the original value
            .alias("unit_sales")
        ]
    )

    df = df.drop(
        "is_missing", "missing_group", "missing_count", "group_max_missing_count"
    )

    # Convert Polars df back to Pandas df
    df = df.to_pandas()

    return df

### 3.2. Aggregate dataset to weekly level

- Group the DataFrame by store number, item number, year, and week_cum_number, then aggregate the columns
--> "unit_sales","onpromotion", "holiday_local_count","holiday_regional_count","holiday_national_count",


In [9]:
def aggregate_week(df):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "year", "week_nbr"])

    # Group by the specified columns and aggregate
    df = (
        df.groupby(
            [
                "store_nbr",
                "item_nbr",
                "year",
                "week_number_cum",  # Aggregating by week_number_cum
            ]
        )
        .agg(
            {
                "unit_sales": "sum",
                "onpromotion": "sum",
                "holiday_local_count": "sum",
                "holiday_regional_count": "sum",
                "holiday_national_count": "sum",
                "date": "first",  # Keep the first day of week, needed to run Timeseries models from SKtime
                "store_type": "first",  # Keep the first occurrence of store_type
                "store_cluster": "first",  # Keep the first occurrence of store_cluster
                "item_family": "first",  # Keep the first occurrence of item_family
                "item_class": "first",  # Keep the first occurrence of item_class
                "perishable": "first",  # Keep the first occurrence of perishable
                "store_status": "last",  # Keep the last occurrence of store_status
                "item_status": "last",  # Keep the last occurrence of item_status
            }
        )
        .reset_index()
    )

    return df

## 4. Pipeline and preprocessing

Splitting and preprocessing with imputation and aggregating to weekly data

In [10]:
# features = [
#     "date",
#     "store_nbr",
#     "item_nbr",  # , 'item_family', 'store_type', 'perishable'
# ]


# target_variable = ["unit_sales"]

In [11]:
features = [
    "store_nbr",
    "item_nbr",
    "date",
    "onpromotion",
    # "holiday_local_count",
    # "holiday_national_count",
    # "holiday_regional_count",
    "store_type",
    "store_cluster",
    "item_family",
    "item_class",
    "perishable",
    # "store_status",
    # "item_status",
    "year",
    "week_number_cum",
]

target_variable = ["unit_sales"]

In [12]:
def impute_agg_preprocessing(df, window_size=7):

    df = impute_stockouts_polars(df, window_size)

    df = aggregate_week(df)

    return df

In [13]:
def preprocess_split_filter(df, features, target_variable):

    # Splitting in train, test, validation split
    print(f"\nStep 1: Splitting in train, test, validation split")
    train_df, test_df, val_df = train_test_val_split(df)

    # Preprocessing with imputation and aggregating to weekly data
    print(f"\nStep 2: Preprocessing with imputation and aggregating to weekly data")
    train_df = impute_agg_preprocessing(train_df)
    test_df = impute_agg_preprocessing(test_df)
    val_df = impute_agg_preprocessing(val_df)

    # Filter spilts on needed feature and target variables
    print(f"\nStep 3: Filter spilts on needed feature and target variables")
    train_df = train_df[features + target_variable]
    test_df = test_df[features + target_variable]
    val_df = val_df[features + target_variable]

    # Ensure df's are sorted by store, item, and date for alignment
    print(f"\nStep 4: Ensure dfs are sorted by store, item, and date for alignment")
    train_df = train_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    test_df = test_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    val_df = val_df.sort_values(by=["store_nbr", "item_nbr", "date"])

    return train_df, test_df, val_df

In [14]:
# # filter for item 457928 in store 30 to debug HW model
# df_final = df_final[(df_final["store_nbr"] == 30) & (df_final["item_nbr"] == 457928)]

In [15]:
df_final = df_final[
    (df_final["store_nbr"] == 30)
]  # & (df_final["item_nbr"] == 103520)]

In [ ]:
train_df, test_df, val_df = preprocess_split_filter(df_final, features, target_variable)

In [ ]:
def stop

### Write to Parquet fil and saves it in output_path

In [17]:
def save_dataframe_to_parquet(df, output_path, file_prefix="Prepped_data"):
    try:
        # Ensure the directory exists
        os.makedirs(output_path, exist_ok=True)

        # Generate today's date for the filename
        today = date.today().strftime("%Y%m%d")

        # Create the full filename with path
        filename = f"{file_prefix}_{today}.parquet"
        full_path = os.path.join(output_path, filename)

        # Save the DataFrame to a Parquet file
        df.to_parquet(full_path)

        print(f"DataFrame successfully saved to {full_path}")

        return full_path

    except Exception as e:
        print(f"Error saving DataFrame to Parquet file: {e}")

        return None

In [18]:
output_path = "C:/Users/alexander/Documents/0. Data Science and AI for Experts/FILE"

In [19]:
# train_df_saved_path = save_dataframe_to_parquet(
#    train_df, output_path, file_prefix="train_df"
# )

## 5. Model Pipeline

### 5.1 Model: Holt-Winters

In [20]:
def holt_winters_model_forecast(
    train_df,
    forecast_df,
    seasonal_periods=None,
    trend=None,
    damped_trend=None,
    seasonal=None,
    smoothing_level=None,
    smoothing_trend=None,
    smoothing_seasonal=None,
    damping_trend=None,
):
    # Unique store and item combinations
    unique_stores = train_df["store_nbr"].unique()

    forecasts = {}
    fitted_values = {}
    model_params = []

    for store in unique_stores:
        print(f"Model Hollt-Winters starts training for store {store}")

        unique_items = train_df["item_nbr"].unique()

        for item in unique_items:
            # Filter the data for the specific store and item
            train_data = train_df[
                (train_df["store_nbr"] == store) & (train_df["item_nbr"] == item)
            ].copy()

            # Convert date to datetime and set as index
            train_data["date"] = pd.to_datetime(train_data["date"])
            train_data = train_data.set_index("date")
            train_data = train_data.asfreq(
                "W-MON"
            )  # Setting frequency to weekly, starting week at Monday

            # Extract the unit_sales series
            train_series = train_data["unit_sales"].copy()

            # Fit Holt-Winters Model
            try:
                model = ExponentialSmoothing(
                    train_series,
                    trend=trend,
                    damped_trend=damped_trend,
                    seasonal=seasonal,
                    seasonal_periods=seasonal_periods,
                )

                fitted_model = model.fit(
                    smoothing_level=smoothing_level,
                    smoothing_trend=smoothing_trend,
                    smoothing_seasonal=smoothing_seasonal,
                    damping_trend=damping_trend,
                )

                # fitted_model = model.fit(optimized=True, use_brute=True)

                # Forecast the forecast period length
                forecast_length = len(
                    forecast_df[
                        (forecast_df["store_nbr"] == store)
                        & (forecast_df["item_nbr"] == item)
                    ]["date"].unique()
                )

                forecast = fitted_model.forecast(forecast_length)

                # Store the forecast
                forecasts[(store, item)] = forecast
                fitted_values[(store, item)] = fitted_model.fittedvalues

                # Extract model parameters
                params = {
                    "store_nbr": store,
                    "item_nbr": item,
                    "damping": fitted_model.params.get("damping_slope", None),
                    "alpha": fitted_model.model.params.get("smoothing_level", None),
                    "beta": fitted_model.model.params.get("smoothing_trend", None),
                    "gamma": fitted_model.model.params.get("smoothing_seasonal", None),
                    "theta": fitted_model.model.params.get("damping_trend", None),
                }
                model_params.append(params)

            except Exception as e:
                print(f"Model failed for store {store}, item {item}: {e}")

    # Convert model parameters to df
    params_df = pd.DataFrame(model_params)

    return forecasts, fitted_values, params_df

### 5.2 Holt Winters with unique_id and optimizer parameter

In [21]:
def holt_winters_model_forecast(
    train_df,
    forecast_df,
    seasonal_periods=None,
    trend=None,
    damped_trend=None,
    seasonal=None,
    smoothing_level=None,
    smoothing_trend=None,
    smoothing_seasonal=None,
    damping_trend=None,
    optimized=False,
    use_brute=False,
):

    forecasts = {}
    fitted_values = {}
    model_params = []
    fitted_models = {}

    # Create and store the current unique identifier for tracking
    train_df["unique_id"] = (
        train_df["store_nbr"].astype(str) + "_" + train_df["item_nbr"].astype(str)
    )

    # Extract all unique identifiers from the unique_id column
    unique_ids = train_df["unique_id"].unique()

    # Loop through each unique id
    for unique_id in unique_ids:

        # Extract store and item numbers
        store, item = unique_id.split("_")

        # Filter the data for the specific unique_id
        train_data = train_df[train_df["unique_id"] == unique_id].copy()

        # Convert date to datetime and set as index
        train_data["date"] = pd.to_datetime(train_data["date"])
        train_data = train_data.set_index("date")
        train_data = train_data.asfreq(
            "W-MON"
        )  # Setting frequency to weekly, starting week at Monday

        # Extract the unit_sales series
        train_series = train_data["unit_sales"].copy()

        # Fit Holt-Winters Model
        try:
            model = ExponentialSmoothing(
                train_series,
                trend=trend,
                damped_trend=damped_trend,
                seasonal=seasonal,
                seasonal_periods=seasonal_periods,
            )

            fitted_model = model.fit(
                smoothing_level=smoothing_level,
                smoothing_trend=smoothing_trend,
                smoothing_seasonal=smoothing_seasonal,
                damping_trend=damping_trend,
                optimized=optimized,
                use_brute=use_brute,
            )

            # Store the fitted model for each unique_id
            fitted_models[unique_id] = fitted_model

            # Forecast the forecast period length
            forecast_length = len(forecast_df["date"].unique())

            forecast = fitted_model.forecast(forecast_length)

            # Store the forecast
            forecasts[(store, item)] = forecast
            fitted_values[(store, item)] = fitted_model.fittedvalues

            # Extract model parameters
            params = {
                "unique_id": unique_id,
                "store_nbr": store,
                "item_nbr": item,
                "damping": fitted_model.params.get("damping_slope", None),
                "alpha": fitted_model.model.params.get("smoothing_level", None),
                "beta": fitted_model.model.params.get("smoothing_trend", None),
                "gamma": fitted_model.model.params.get("smoothing_seasonal", None),
                "theta": fitted_model.model.params.get("damping_trend", None),
            }
            model_params.append(params)

        except Exception as e:
            print(f"Model failed for unique_id {unique_id}: {e}")

    # Convert model parameters to df
    params_df = pd.DataFrame(model_params)

    return forecasts, fitted_values, params_df, fitted_models

In [22]:
# train_val_df = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
# Run model with Optimized=True,  use_brute=True
test_forecasts, train_fitted_values, train_params_df, fitted_models = (
    holt_winters_model_forecast(
        train_df,
        test_df,
        seasonal_periods=52,
        trend="add",
        damped_trend=True,
        seasonal="add",
        optimized=True,
        use_brute=True,
    )
)

In [ ]:
DEF STOP

In [222]:
# Run model with manual params
test_forecasts, train_fitted_values, train_params_df = holt_winters_model_forecast(

    train_df,
    test_df,
    seasonal_periods=52,
    trend="add",
    damped_trend=True,
    seasonal="add",
    smoothing_level=0.2,
    smoothing_trend=0,
    smoothing_seasonal=0,
    damping_trend=0,
)

In [ ]:
train_params_df.head(5)

### 5.3 Model Fit seperated from Predicting/Forecasting function

#### 5.3.1. Model Fit

In [24]:
def holt_winters_model_fit(
    train_df,
    forecast_df,
    seasonal_periods=None,
    trend=None,
    damped_trend=None,
    seasonal=None,
    smoothing_level=None,
    smoothing_trend=None,
    smoothing_seasonal=None,
    damping_trend=None,
    optimized=False,
    use_brute=False,
):

    model_params = []
    fitted_models = {}

    # Create and store the current unique identifier for tracking
    train_df["unique_id"] = (
        train_df["store_nbr"].astype(str) + "_" + train_df["item_nbr"].astype(str)
    )

    # Extract all unique identifiers from the unique_id column
    unique_ids = train_df["unique_id"].unique()

    # Loop through each unique id
    for unique_id in unique_ids:

        # Extract store and item numbers
        store, item = unique_id.split("_")

        # Filter the data for the specific unique_id
        train_data = train_df[train_df["unique_id"] == unique_id].copy()

        # Convert date to datetime and set as index
        train_data["date"] = pd.to_datetime(train_data["date"])
        train_data = train_data.set_index("date")
        train_data = train_data.asfreq(
            "W-MON"
        )  # Setting frequency to weekly, starting week at Monday

        # Extract the unit_sales series
        train_series = train_data["unit_sales"].copy()

        # Fit Holt-Winters Model
        try:
            model = ExponentialSmoothing(
                train_series,
                trend=trend,
                damped_trend=damped_trend,
                seasonal=seasonal,
                seasonal_periods=seasonal_periods,
            )

            fitted_model = model.fit(
                smoothing_level=smoothing_level,
                smoothing_trend=smoothing_trend,
                smoothing_seasonal=smoothing_seasonal,
                damping_trend=damping_trend,
                optimized=optimized,
                use_brute=use_brute,
            )

            # Store the fitted model for each unique_id
            fitted_models[unique_id] = fitted_model

            # Extract model parameters
            params = {
                "unique_id": unique_id,
                "store_nbr": store,
                "item_nbr": item,
                "damping": fitted_model.params.get("damping_slope", None),
                "alpha": fitted_model.model.params.get("smoothing_level", None),
                "beta": fitted_model.model.params.get("smoothing_trend", None),
                "gamma": fitted_model.model.params.get("smoothing_seasonal", None),
                "theta": fitted_model.model.params.get("damping_trend", None),
            }
            model_params.append(params)

        except Exception as e:
            print(f"Model failed for unique_id {unique_id}: {e}")

    # Convert model parameters to df
    params_df = pd.DataFrame(model_params)
    params_df["store_nbr"] = params_df["store_nbr"].astype("int8")
    params_df["item_nbr"] = params_df["item_nbr"].astype("int32")

    return params_df, fitted_models

In [25]:
# Run model with Optimized=True,  use_brute=True
train_params_df, fitted_models = holt_winters_model_fit(
    train_df,
    test_df,
    seasonal_periods=52,
    trend="add",
    damped_trend=True,
    seasonal="add",
    optimized=True,
    use_brute=True,
)

#### 5.3.2. Forecast and predicting values

In [26]:
def holt_winters_forecast(
    forecast_df,
    # params_df,
    fitted_models,
):

    forecasts = {}

    # Create and store the current unique identifier for tracking
    forecast_df["unique_id"] = (
        forecast_df["store_nbr"].astype(str) + "_" + train_df["item_nbr"].astype(str)
    )

    # Extract all unique identifiers from the unique_id column
    unique_ids = forecast_df["unique_id"].unique()

    # Loop through each unique id
    for unique_id in unique_ids:

        # Extract store and item numbers
        store, item = unique_id.split("_")

        # Extract the pre-trained model  for the specific unique_id
        fitted_model = fitted_models[unique_id]

        # Forecast with pre-trained Holt-Winters model
        try:

            # Forecast the forecast period length
            forecast_length = len(forecast_df["date"].unique())
            forecast = fitted_model.forecast(forecast_length)

            # Store the forecast
            forecasts[(store, item)] = forecast

        except Exception as e:
            print(f"Model failed for unique_id {unique_id}: {e}")

    return forecasts

In [27]:
test_forecasts = holt_winters_forecast(test_df, fitted_models)

### 5.4. Evaulation Metrics and Evaluate Model functions

In [83]:
def calculate_metrics(y_true, y_pred):

    y_true = np.array(y_true)  # Convert to NumPy array
    y_pred = np.array(y_pred)  # Convert to NumPy array

    mape = mean_absolute_percentage_error(y_true, y_pred)

    accuracy = 1 - mape

    bias = np.mean(y_pred - y_true)

    mse = np.mean(([y_true] - [y_pred]) ** 2)  # Mean Squared Error
    rmse = np.sqrt(mse)  # Root Mean Squared Error

    return {"RMSE": rmse, "MAPE": mape, "Accuracy": accuracy, "Bias": bias}

In [ ]:
def evaluate_forecasts(forecasts, forcast_df):

    # Initialize lists to store metrics for each store and item
    all_metrics = []
    store_metrics = {}

    # Iterate over each store and item combination to collect true and predicted values
    for (store, item), forecast in forecasts.items():

        y_true = forcast_df[
            (forcast_df["store_nbr"] == store) & (forcast_df["item_nbr"] == item)
        ]["unit_sales"]
        y_pred = forecast

        # Index the values
        # y_true = y_true.reset_index(drop=True)
        # y_pred = pd.Series(forecast).reset_index(drop=True)

        if len(y_true) == 0 or len(y_pred) == 0:
            print(f"Skipping store {store}, item {item} due to empty data.")
            continue

        metrics = calculate_metrics(y_true, y_pred)
        metrics.update({"store_nbr": store, "item_nbr": item})
        all_metrics.append(metrics)

        # Initialize metrics for the store if not already present
        if store not in store_metrics:
            store_metrics[store] = {'RMSE': [], "MAPE": [], "Accuracy": [], "Bias": []}

        store_metrics[store]["RMSE"].append(metrics["RMSE"])
        store_metrics[store]["MAPE"].append(metrics["MAPE"])
        store_metrics[store]["Accuracy"].append(metrics["Accuracy"])
        store_metrics[store]["Bias"].append(metrics["Bias"])

    # Calculate average metrics for each store
    average_store_metrics = []
    for store, metrics in store_metrics.items():
        
        average_rmse = np.nanmean(metrics["RMSE"])
        average_mape = np.nanmean(metrics["MAPE"])
        average_accuracy = np.nanmean(metrics["Accuracy"])
        average_bias = np.nanmean(metrics["Bias"])

        average_store_metrics.append(
            {
                "store_nbr": store,
                "item_nbr": "average",
                "RMSE" : average_rmse,
                "MAPE": average_mape,
                "Accuracy": average_accuracy,
                "Bias": average_bias,
            }
        )

    # Calculate overall average metrics
    overall_rmse = np.nanmean(
        [metric["RMSE"] for metric in all_metrics if not np.isnan(metric["RMSE"])]
    )
    
    overall_mape = np.nanmean(
        [metric["MAPE"] for metric in all_metrics if not np.isnan(metric["MAPE"])]
    )
    overall_accuracy = np.nanmean(
        [
            metric["Accuracy"]
            for metric in all_metrics
            if not np.isnan(metric["Accuracy"])
        ]
    )
    overall_bias = np.nanmean(
        [metric["Bias"] for metric in all_metrics if not np.isnan(metric["Bias"])]
    )

    overall_metrics = {
        "store_nbr": "overall",
        "item_nbr": "overall",
        "RMSE" : average_rmse
        "MAPE": overall_mape,
        "Accuracy": overall_accuracy,
        "Bias": overall_bias,
    }

    # Convert metrics to df's
    metrics_df = pd.DataFrame(all_metrics)
    average_store_metrics_df = pd.DataFrame(average_store_metrics)
    overall_metrics_df = pd.DataFrame([overall_metrics])

    # Print metrics
    # print(metrics_df)
    # print(average_store_metrics_df)
    print(overall_metrics_df)

    return metrics_df, average_store_metrics_df, overall_metrics_df

In [ ]:
# # Evaluate forecasts and print metrics
# test_metrics_df, test_average_store_metrics_df, test_df_overall_metrics_df = (
#     evaluate_forecasts(test_forecasts, test_df)
# )

In [29]:
df_test_true_pred = add_forecasts_to_df(test_df, test_forecasts)

In [31]:
# Example columns: 'actual' and 'predicted'
def calculate_rmse(df, actual_col, predicted_col):
    mse = np.mean((df[actual_col] - df[predicted_col]) ** 2)  # Mean Squared Error
    rmse = np.sqrt(mse)  # Root Mean Squared Error

    return rmse

In [ ]:
rmse = calculate_rmse(df_test_true_pred, "y_true", "y_pred")
print(f"RMSE: {rmse}")

- RMSE store 1, step 0: 18.587704313442728
- RMSE naive params  : 60.99798457949619
- RMSE store 1, step 2: 13.131692612885391
- RMSE store 1, optimized params : 

In [ ]:
train_params_df_x = train_params_df[(train_params_df["item_nbr"] == 108797)]
train_params_df_x.head()

In [35]:
df_test_true_pred_x = df_test_true_pred[(df_test_true_pred["item_nbr"] == 108797)]

In [ ]:
# Plotting y_true and y_pred over time
plt.figure(figsize=(14, 7))
plt.plot(
    df_test_true_pred_x["date"],
    df_test_true_pred_x["y_true"],
    label="y_true (Actual)",
    marker="o",
    linestyle="-",
    color="blue",
)
plt.plot(
    df_test_true_pred_x["date"],
    df_test_true_pred_x["y_pred"],
    label="y_pred (Predicted)",
    marker="x",
    linestyle="--",
    color="red",
)

# Adding labels, title, and legend
plt.title("Comparison of Actual (y_true) and Predicted (y_pred) Values Over Time")
plt.xlabel("Date")
plt.ylabel("Values")
plt.legend()
plt.grid()
plt.show()

In [ ]:
test_average_store_metrics_df

In [ ]:
test_metrics_df.sort_values(by="MAPE", ascending=True).head(30)

### 5.X Function to join y_pred to Y_ptrue in original df

In [28]:
def add_forecasts_to_df(original_df, forecasts):

    # Rename unit_sales to y_true
    original_df = original_df.rename(columns={"unit_sales": "y_true"})

    # Ensure the original DataFrame has a datetime index
    original_df["date"] = pd.to_datetime(original_df["date"])
    original_df = original_df.set_index(["date", "store_nbr", "item_nbr"])

    # Prepare a DataFrame for forecasts
    forecast_entries = []

    for (store, item), forecast_series in forecasts.items():
        # Convert Forecast servies to df
        forecast_df = forecast_series.reset_index()

        # Rename columns to y_pred and add store and item columns
        forecast_df.columns = ["date", "y_pred"]
        forecast_df["store_nbr"] = store
        forecast_df["item_nbr"] = item
        forecast_entries.append(forecast_df)

    # Combine all forecast entries
    forecast_df = pd.concat(forecast_entries, ignore_index=True)
    forecast_df["date"] = pd.to_datetime(forecast_df["date"])

    forecast_df["store_nbr"] = forecast_df["store_nbr"].astype("int8")
    forecast_df["item_nbr"] = forecast_df["item_nbr"].astype("int32")

    forecast_df = forecast_df.set_index(["date", "store_nbr", "item_nbr"])

    # Join the forecasted values with the original df
    result_df = original_df.join(
        forecast_df, on=["date", "store_nbr", "item_nbr"], how="left"
    )

    result_df = result_df.reset_index()

    # Specify the desired column order
    desired_order = [
        "date",
        "week_number_cum",
        "store_nbr",
        "item_nbr",
        "y_true",
        "y_pred",
        "perishable",
        "store_type",
        "store_cluster",
        "item_family",
        "item_class",
    ]
    # Add the rest of the columns to the desired order
    all_columns = list(result_df.columns)
    remaining_columns = [col for col in all_columns if col not in desired_order]
    final_order = desired_order + remaining_columns

    # Reorder the DataFrame columns
    result_df = result_df[final_order]

    return result_df

In [ ]:
df_test_with_forecasts, test_forecast_df = add_forecasts_to_df(test_df, test_forecasts)

df_test_with_forecasts = save_dataframe_to_parquet(
    df_test_with_forecasts, output_path, file_prefix="df_test_with_forecasts_optimized"
)

----------------------

In [ ]:
train_df_457928 = train_df[(train_df["item_nbr"] == 457928)]

In [149]:
train_df_1 = train_df[(train_df["store_nbr"] == 1)]

-------------------------------------------